<a href="https://colab.research.google.com/github/anathayna/tcc/blob/main/tcc_fim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="orange">**TCC: Identificação do discurso de ódio em memes**</font>

O Google Colab é uma plataforma baseada em Jupyter Notebook que oferece um ambiente de execução em nuvem com recursos gratuitos de GPU e TPU, amplamente utilizado em projetos de aprendizado de máquina e ciência de dados.

Este notebook documenta a etapa final do desenvolvimento do trabalho de conclusão de curso, cujo objetivo é identificar e classificar discursos de ódio em memes por meio de técnicas de aprendizado de máquina multimodal.

# <font color="orange">**Sumário**</font>

1.   Bibliotecas e dependências
2.   Banco de dados
4.   Treinamento
5.   Classificação
6.   Avaliação
7.   Conclusão

## <font color="orange">1. Bibliotecas e dependências</font>

In [1]:
!pip install -q ftfy regex tqdm jsonlines open-clip-torch
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q --upgrade torch torchvision torchaudio

  Preparing metadata (setup.py) ... done


In [2]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [3]:
!mamba install -c pytorch -c nvidia -c conda-forge faiss-gpu=1.7.4 mkl=2021 blas=1.0=mkl -y


Looking for: ['faiss-gpu=1.7.4', 'mkl=2021', 'blas==1.0=mkl']

warning  libmamba Could not parse mod/etag header
[+] 0.0s
pytorch/linux-64 (..  ⣾  pytorch/linux-64 (check zst)                      
warning  libmamba Could not parse mod/etag header
[+] 0.0s
pytorch/noarch (ch..  ⣾  pytorch/noarch (check zst)                        
warning  libmamba Could not parse mod/etag header
[+] 0.0s
nvidia/linux-64 (c..  ⣾  nvidia/linux-64 (check zst)                       
warning  libmamba Could not parse mod/etag header
[+] 0.0s
nvidia/noarch (che..  ⣾  nvidia/noarch (check zst)                         
warning  libmamba Could not parse mod/etag header
warning  libmamba Could not parse mod/etag header
[+] 0.0s
pytorch/linux-64  ⣾  pytorch/linux-64                                  
pytorch/noarch                                      10.5kB @ 119.9kB/s  0.1s
nvidia/noarch                                       47.4kB @ 508.1kB/s  0.1s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  


In [4]:
import os

repo_path = '/content/drive/MyDrive/RGCL'

if not os.path.exists(repo_path):
    !git clone https://github.com/JingbiaoMei/RGCL.git {repo_path}
else:
    print("Repositório já existe no Drive. Pulando o clone.")

%cd {repo_path}

!pip install -r requirement.txt

Cloning into '/content/drive/MyDrive/RGCL'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 225 (delta 27), reused 28 (delta 24), pack-reused 193 (from 1)
Receiving objects: 100% (225/225), 111.04 KiB | 4.27 MiB/s, done.
Resolving deltas: 100% (122/122), done.
/content/drive/MyDrive/RGCL
  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [5]:
import os
import shutil
import glob
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import clip
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive

## <font color="orange">2. Banco de dados</font>

In [6]:
#@markdown Defina o caminho para o arquivo **.zip** do banco de dados do *Hateful Memes*.
#@markdown **exemplo:** `"/content/drive/MyDrive/hateful_memes.zip"`

PATH_TO_ZIP_FILE = '/content/drive/MyDrive/hateful_memes.zip' #@param {type:"string"}

#@markdown Defina o diretório base para extrair o banco de dados.
#@markdown **exemplo:** `"/content"`

HOME = '/content' #@param {type:"string"}

In [7]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
os.chdir(HOME)
os.getcwd()
os.environ['PYTHONPATH'] += ":/content/model/"

In [9]:
drive.mount('/content/drive', force_remount=True)

zip_path = PATH_TO_ZIP_FILE
extract_path = '/content/model/'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


Mounted at /content/drive


In [10]:
img_dir = '/content/model/hateful_memes/img'

image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp']

image_count = 0
for filename in os.listdir(img_dir):
    if any(filename.lower().endswith(ext) for ext in image_extensions):
        image_count += 1

print(f"total de imagens: {image_count}")

total de imagens: 12140


In [11]:
rgcl_data_dir = '/content/RGCL/data/image/FB/All/'

img_dir = '/content/model/hateful_memes/img'

os.makedirs(rgcl_data_dir, exist_ok=True)

print("Copiando imagens para a estrutura do RGCL...")
images = [f for f in os.listdir(img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
for img_file in tqdm(images):
    src = os.path.join(img_dir, img_file)
    dst = os.path.join(rgcl_data_dir, img_file)
    shutil.copy(src, dst)

print(f"Total de {len(images)} imagens copiadas para {rgcl_data_dir}")

Copiando imagens para a estrutura do RGCL...


  0%|          | 0/12140 [00:00<?, ?it/s]

Total de 12140 imagens copiadas para /content/RGCL/data/image/FB/All/


In [12]:
rgcl_gt_dir = '/content/RGCL/data/gt/FB/'
os.makedirs(rgcl_gt_dir, exist_ok=True)

base_path = '/content/model/hateful_memes/'

jsonl_files = glob.glob(os.path.join(base_path, '*.jsonl'))

print(f"Copiando arquivos de anotação para {rgcl_gt_dir}...")
for file_path in jsonl_files:
    file_name = os.path.basename(file_path)
    dst_path = os.path.join(rgcl_gt_dir, file_name)
    shutil.copy(file_path, dst_path)
    print(f"Copiado: {file_name}")

print("Processo concluído!")

Copiando arquivos de anotação para /content/RGCL/data/gt/FB/...
Copiado: test_seen.jsonl
Copiado: train.jsonl
Copiado: dev_unseen.jsonl
Copiado: dev_seen.jsonl
Copiado: test_unseen.jsonl
Processo concluído!


In [13]:
%cd /content/RGCL
!python3 src/utils/generate_CLIP_embedding_HF.py --dataset "FB"

/content/RGCL
Namespace(EXP_FOLDER='./data/CLIP_Embedding', model='openai/clip-vit-large-patch14-336', image_size=336, dataset='FB', batch_size=32, all=False)
Loading weights: 100% 391/391 [00:00<00:00, 26309.87it/s]
[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight       

In [14]:
%cd /content/RGCL
!python3 src/utils/generate_ALIGN_embedding_HF.py --dataset "FB"

/content/RGCL
Namespace(EXP_FOLDER='./data/CLIP_Embedding', model='kakaobrain/align-base', image_size=346, dataset='FB', batch_size=32, all=False, trunc=True, token_paral=False)
config.json: 5.25kB [00:00, 1.53MB/s]
pytorch_model.bin: 100% 690M/690M [00:05<00:00, 136MB/s]
model.safetensors:   0% 0.00/690M [00:00<?, ?B/s]
Loading weights: 100% 1394/1394 [00:00<00:00, 19285.36it/s]
model.safetensors:   0% 0.00/690M [00:00<?, ?B/s]
tokenizer_config.json: 100% 399/399 [00:00<00:00, 1.69MB/s]
model.safetensors:   0% 0.00/690M [00:00<?, ?B/s]
vocab.txt: 232kB [00:00, 11.7MB/s]
model.safetensors:   0% 0.00/690M [00:01<?, ?B/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 258kB/s]
model.safetensors:   0% 0.00/690M [00:01<?, ?B/s]
preprocessor_config.json: 100% 508/508 [00:00<00:00, 1.27MB/s]
model.safetensors:  32% 219M/690M [00:02<00:03, 148MB/s] /usr/local/lib/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 24 worker processes in t

In [15]:
%cd /content/RGCL
!MPLBACKEND=agg bash scripts/experiments.sh

/content/RGCL
Namespace(path='./data/', output_path='./logging/Retrieval/FB/RAC/RAC_lr0.0001_Bz64_Ep30_cosSim_triplet_drop[0.2, 0.4, 0.1]_topK20__PseudoGold_positive_1_hard_negative_1_seed0_hybrid_loss/', model='openai_clip-vit-large-patch14-336_HF', dataset='FB', similarity_threshold=-1.0, fusion_mode='align', topk=20, majority_voting='arithmetic', metric='cos', loss='triplet', triplet_margin=0.1, norm_feats_loss=False, l2_sqrt=False, hybrid_loss=True, ce_weight=0.5, pos_weight_value=None, num_layers=3, proj_dim=1024, map_dim=1024, dropout=[0.2, 0.4, 0.1], batch_norm=False, last_layer='none', epochs=30, batch_size=64, lr=0.0001, weight_decay=0.0001, lr_scheduler=False, num_workers=24, grad_clip=0.1, no_pseudo_gold_positives=1, in_batch_loss=True, hard_negatives_loss=True, no_hard_negatives=1, no_hard_positives=0, hard_negatives_multiple=12, Faiss_GPU=True, reindex_every_step=False, sparse_dictionary=None, use_attribute=True, sparse_topk=None, eval_retrieval=True, log_interval=10, fina

## <font color="orange">4. Treinamento</font>

## <font color="orange">5. Classificação</font>

## <font color="orange">6. Avaliação</font>

## <font color="orange">7. Conclusão</font>

## <font color="orange">Teste</font>